# Phase 4 : Spark Processing and Analytics Layer
**Domain:** Customer Service : CFPB Consumer Complaints  
**Team:** Kennedy (Student B), Mahdi (Student C), Max (Student A)  
**Date:** April 2026  

**Goal:** Transition from our Phase 3 PostgreSQL star schema to a distributed PySpark processing engine.  
We load the raw dataframe downloaded from the website, split into our 4 normalized CSV tables, enforce strict schemas, clean and enrich the data, run analytical Spark SQL queries, and persist outputs in Parquet and CSV formats for the gold medallion structure.

In [1]:
#Importing the Raw Data (Bronze Medallion)

# Create the Spark Session (entry point to the Spark engine)

from google.colab import drive
drive.mount('/content/drive')
print("Drive Mounted")


import os, shutil
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import (
    col, explode_outer, count, desc, regexp_replace,
    trim, when, lit, datediff, split, explode,
    coalesce, to_date, length
)
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType,
    DateType, BooleanType
)


RAW_CSV    = r"/content/drive/MyDrive/complaints.csv"
OUTPUT_DIR = r"/content/drive/MyDrive/phase4_parquet_output"

#Starting Spark Session
spark = (SparkSession.builder
         .appName("CFPB_Phase2_Split")
         .master("local[*]")
         .config("spark.driver.memory", "8g")
         .getOrCreate())

#Loading Raw dataframe
print("Reading raw CSV (with multiLine to handle embedded newlines) …")
raw = (spark.read
       .option("header", "true")
       .option("multiLine", "true")
       .option("escape", '"')
       .option("inferSchema", "true")
       .csv(RAW_CSV))
total = raw.count()
print(f"Total rows: {total:,}")

raw = (raw
       .withColumnRenamed("Complaint ID",                "complaint_id")
       .withColumnRenamed("Date received",               "date_received")
       .withColumnRenamed("Product",                     "product_name")
       .withColumnRenamed("Sub-product",                 "sub_product_name")
       .withColumnRenamed("Issue",                       "issue_name")
       .withColumnRenamed("Sub-issue",                   "sub_issue_name")
       .withColumnRenamed("Company",                     "company_name")
       .withColumnRenamed("Company public response",     "company_response")
       .withColumnRenamed("State",                       "state")
       .withColumnRenamed("ZIP code",                    "zip_code")
       .withColumnRenamed("Submitted via",               "submitted_via")
       .withColumnRenamed("Date sent to company",        "date_sent_to_company")
       .withColumnRenamed("Consumer consent provided?",  "consumer_consent")
       .withColumnRenamed("Timely response?",            "timely_response")
       .withColumnRenamed("Consumer disputed?",          "consumer_disputed")
       .withColumnRenamed("Tags",                        "tags")
       .withColumnRenamed("Consumer complaint narrative","complaint_narrative")
       )

# ── DIMENSION 1: products ────────────────────────────────────────────────────
products_raw = (raw
            .select("product_name", "sub_product_name")
            .dropDuplicates()
            .withColumn("product_id",
                        F.row_number().over(
                            Window.orderBy("product_name", "sub_product_name")))
            )
print(f"products dimension: {products_raw.count()} rows")

# ── DIMENSION 2: issues ──────────────────────────────────────────────────────
issues_raw = (raw
          .select("issue_name", "sub_issue_name")
          .dropDuplicates()
          .withColumn("issue_id",
                      F.row_number().over(
                          Window.orderBy("issue_name", "sub_issue_name")))
          )
print(f"issues dimension: {issues_raw.count()} rows")

# ── DIMENSION 3: companies ───────────────────────────────────────────────────
# Standardize: uppercase + trim to merge case variations
raw = raw.withColumn("company_name_clean",
                     F.upper(F.trim(F.col("company_name"))))

companies_raw = (raw
             .select("company_name_clean")
             .dropDuplicates()
             .withColumnRenamed("company_name_clean", "company_name")
             .withColumn("company_id",
                         F.row_number().over(
                             Window.orderBy("company_name")))
             )
print(f"companies dimension: {companies_raw.count()} rows")

# ── FACT TABLE: complaints ───────────────────────────────────────────────────
complaints_raw = (raw
              .join(products_raw,  ["product_name", "sub_product_name"], "left")
              .join(issues_raw,    ["issue_name",   "sub_issue_name"],   "left")
              .join(companies_raw, raw["company_name_clean"] == companies_raw["company_name"], "left")
              .select(
                  F.col("complaint_id").cast("int"),
                  "product_id",
                  "issue_id",
                  "company_id",
                  F.to_date("date_received", "MM/dd/yyyy").alias("date_received"),
                  F.to_date("date_sent_to_company", "MM/dd/yyyy").alias("date_sent_to_company"),
                  "state",
                  "zip_code",
                  "submitted_via",
                  "consumer_consent",
                  F.when(F.col("timely_response") == "Yes", True)
                   .otherwise(False).alias("timely_response"),
                  F.when(F.col("consumer_disputed") == "Yes", True)
                   .otherwise(False).alias("consumer_disputed"),
                  "tags",
                  "company_response",
                  "complaint_narrative"
              ))
print(f"complaints fact: {complaints_raw.count():,} rows")

#Cleaning the Messy Data and enforcing schemas (Silver Medalion)

# Show the inferred schema BEFORE enforcement
print("=== INFERRED SCHEMA (complaints) ===")
complaints_raw.printSchema()



complaints_schema = StructType([
    StructField("complaint_id",         IntegerType(), False),
    StructField("product_id",           IntegerType(), True),
    StructField("issue_id",             IntegerType(), True),
    StructField("company_id",           IntegerType(), True),
    StructField("date_received",        DateType(),    True),
    StructField("date_sent_to_company", DateType(),    True),
    StructField("state",                StringType(),  True),
    StructField("zip_code",             StringType(),  True),
    StructField("submitted_via",        StringType(),  True),
    StructField("consumer_consent",     StringType(),  True),
    StructField("timely_response",      BooleanType(), True),
    StructField("consumer_disputed",    BooleanType(), True),
    StructField("tags",                 StringType(),  True),
    StructField("company_response",     StringType(),  True),
    StructField("complaint_narrative",  StringType(),  True),
])

products_schema = StructType([
    StructField("product_id",       IntegerType(), False),
    StructField("product_name",     StringType(),  True),
    StructField("sub_product_name", StringType(),  True),
])

issues_schema = StructType([
    StructField("issue_id",       IntegerType(), False),
    StructField("issue_name",     StringType(),  True),
    StructField("sub_issue_name", StringType(),  True),
])

companies_schema = StructType([
    StructField("company_id",   IntegerType(), False),
    StructField("company_name", StringType(),  False),
])

# Re-initialize with strict schemas enforced
complaints = complaints_raw.select([
    col(field.name).cast(field.dataType).alias(field.name)
    for field in complaints_schema.fields
])
products   = products_raw.select([
    col(field.name).cast(field.dataType).alias(field.name)
    for field in products_schema.fields
])
issues     = issues_raw.select([
    col(field.name).cast(field.dataType).alias(field.name)
    for field in issues_schema.fields
])
companies  = companies_raw.select([
    col(field.name).cast(field.dataType).alias(field.name)
    for field in companies_schema.fields
])

print("Schemas Enforsed")
complaints.printSchema()

# Type Conversion & Casting
# Use regexp_replace to clean zip_code (remove any non-digit characters)
# and trim/clean state codes


complaints_clean = complaints \
    .withColumn(
        "zip_code_clean",
        regexp_replace(col("zip_code"), "[^0-9]", "")
    ) \
    .withColumn(
        "state_clean",
        trim(regexp_replace(col("state"), "[^A-Z]", ""))
    )

# Safe Casting with try_cast / validation
# Use regex validation to identify and safely handle invalid zip codes
# Valid US zip: exactly 5 digits

complaints_clean = complaints_clean \
    .withColumn(
        "zip_valid",
        when(
            col("zip_code_clean").rlike("^[0-9]{5}$"), col("zip_code_clean")
        ).otherwise(lit(None))
    ) \
    .withColumn(
        "product_id_safe",
        col("product_id").cast("int")  # try_cast equivalent: nulls on failure
    )

spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")


#  Data Enrichment: Create new calculated columns
# 1. response_days: days between date_received and date_sent_to_company
# 2. response_band: classify response time into categories using WHEN logic

complaints_enriched = complaints_clean \
    .withColumn(
        "response_days",
        datediff(col("date_sent_to_company"), col("date_received"))
    ) \
    .withColumn(
        "response_band",
        when(col("response_days") == 0, "Same Day")
        .when(col("response_days") <= 3, "1-3 Days")
        .when(col("response_days") <= 7, "4-7 Days")
        .when(col("response_days") <= 14, "8-14 Days")
        .when(col("response_days") <= 30, "15-30 Days")
        .when(col("response_days") > 30, "Over 30 Days")
        .otherwise("Unknown")
    ) \
    .withColumn(
        "year_received",
        col("date_received").cast("string").substr(1, 4).cast("int")
    )

# Demonstrate split + explode on the tags column
tags_exploded = complaints_enriched \
    .withColumn("tag_array", split(col("tags"), ",\\s*")) \
    .withColumn("individual_tag", explode_outer(col("tag_array")))

# Silver Medation - Write Cleaned Tables as Parquet and CSV
print("Saving Cleaned Tables to Silver Medallion Folder")

# Convert dates to strings to avoid parsing errors during write
complaints_to_write = complaints_enriched \
    .withColumn("date_received", col("date_received").cast("string")) \
    .withColumn("date_sent_to_company", col("date_sent_to_company").cast("string"))

# Write Silver Medallion as Parquet
complaints_to_write.write.mode("overwrite").parquet("/content/drive/MyDrive/Phase4_Outputs/Silver_Medallion/complaints_parquet")
products.write.mode("overwrite").parquet("/content/drive/MyDrive/Phase4_Outputs/Silver_Medallion/products_parquet")
issues.write.mode("overwrite").parquet("/content/drive/MyDrive/Phase4_Outputs/Silver_Medallion/issues_parquet")
companies.write.mode("overwrite").parquet("/content/drive/MyDrive/Phase4_Outputs/Silver_Medallion/companies_parquet")

print("Silver Medallion Tables written to Parquet")

# Write Silver Medallion as CSV
complaints_to_write.coalesce(1).write.mode("overwrite").option("header", "true").csv("/content/drive/MyDrive/Phase4_Outputs/Silver_Medallion/complaints_csv")
products.coalesce(1).write.mode("overwrite").option("header", "true").csv("/content/drive/MyDrive/Phase4_Outputs/Silver_Medallion/products_csv")
issues.coalesce(1).write.mode("overwrite").option("header", "true").csv("/content/drive/MyDrive/Phase4_Outputs/Silver_Medallion/issues_csv")
companies.coalesce(1).write.mode("overwrite").option("header", "true").csv("/content/drive/MyDrive/Phase4_Outputs/Silver_Medallion/companies_csv")

print("Silver Medallion Tables written to CSV")


#Gold Medallion - Creating Aggrigate tables and Tables that answer KPIs


#Combined Aggregate Table
combined_table = complaints_enriched.join(products,  "product_id", "left") \
              .join(issues,    "issue_id",   "left") \
              .join(companies, "company_id", "left")
print("All Clean Tables Aggrigated")

# Top 10 companies by complaint volume


top_companies_df = complaints_enriched \
    .join(companies, "company_id") \
    .groupBy("company_name") \
    .agg(count("complaint_id").alias("total_complaints")) \
    .orderBy(desc("total_complaints")) \
    .limit(10)
print("Top Companies Dataframe Created")


#Saving the Gold Medallion Dataframes to CSV and Parquet

combined_table.write.mode("overwrite").parquet("/content/drive/MyDrive/Phase4_Outputs/Gold_Medallion/CombinedTable_parquet")
print("Parquet Combined Table saved to drive")
combined_table.coalesce(1).write.mode("overwrite").option("header", "true").csv("/content/drive/MyDrive/Phase4_Outputs/Gold_Medallion/CombinedTable_csv")
print("CSV Combined Table saved to drive")
top_companies_df.write.mode("overwrite").parquet("/content/drive/MyDrive/Phase4_Outputs/Gold_Medallion/TopComplaintCompanies_parquet")
print("Parquet Top Complaint Companies saved to drive")
top_companies_df.coalesce(1).write.mode("overwrite").option("header", "true").csv("/content/drive/MyDrive/Phase4_Outputs/Gold_Medallion/TopComplaintCompanies_csv")
print("CSV Top Complaint Companies saved to drive")

# Stop Spark session
spark.stop()
print("Spark session stopped. Phase 4 complete.")





Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive Mounted
Reading raw CSV (with multiLine to handle embedded newlines) …
Total rows: 14,350,572
products dimension: 127 rows
issues dimension: 441 rows
companies dimension: 7830 rows
complaints fact: 14,350,572 rows
=== INFERRED SCHEMA (complaints) ===
root
 |-- complaint_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- issue_id: integer (nullable = true)
 |-- company_id: integer (nullable = true)
 |-- date_received: date (nullable = true)
 |-- date_sent_to_company: date (nullable = true)
 |-- state: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- submitted_via: string (nullable = true)
 |-- consumer_consent: string (nullable = true)
 |-- timely_response: boolean (nullable = false)
 |-- consumer_disputed: boolean (nullable = false)
 |-- tags: string (nullable = true)
 |-- company_response: string (nullabl